In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

DATA_PATH = Path("../data/raw/sparkov/fraudTrain.csv")

df = pd.read_csv(DATA_PATH)

# Remove original CSV index
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

# Convert timestamp
df["transaction_time"] = pd.to_datetime(
    df["trans_date_trans_time"]
)

# Sort chronologically
df = df.sort_values("transaction_time").reset_index(drop=True)

print("Shape:", df.shape)
print(df["transaction_time"].min())
print(df["transaction_time"].max())

Shape: (1296675, 23)
2019-01-01 00:00:18
2020-06-21 12:13:37


In [3]:
df["hour"] = df["transaction_time"].dt.hour
df["day_of_week"] = df["transaction_time"].dt.dayofweek
df["month"] = df["transaction_time"].dt.month
df["day_of_month"] = df["transaction_time"].dt.day
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

In [4]:
df["log_amount"] = np.log1p(df["amt"])

In [5]:
df["customer_prev_count"] = (
    df.groupby("cc_num")
      .cumcount()
)

In [6]:
df["customer_prev_avg_amount"] = (
    df.groupby("cc_num")["amt"]
      .transform(lambda x: x.shift().expanding().mean())
)

In [7]:
df["customer_prev_median_amount"] = (
    df.groupby("cc_num")["amt"]
      .transform(lambda x: x.shift().expanding().median())
)

In [8]:
df["customer_prev_std_amount"] = (
    df.groupby("cc_num")["amt"]
      .transform(lambda x: x.shift().expanding().std())
)

In [9]:
df.groupby("cc_num")["amt"].mean()

cc_num
60416207185             56.023366
60422928733             69.000784
60423098130            115.046333
60427851591            111.987898
60487002085             50.726028
                          ...    
4958589671582726883     66.377839
4973530368125489546     78.373288
4980323467523543940     74.436429
4989847570577635369     87.582542
4992346398065154184     67.843832
Name: amt, Length: 983, dtype: float64

In [10]:
df["amount_deviation_ratio"] = (
    df["amt"] /
    df["customer_prev_avg_amount"]
)

In [11]:
df[
    [
        "cc_num",
        "transaction_time",
        "amt",
        "customer_prev_avg_amount",
        "amount_deviation_ratio"
    ]
].head(20)

,cc_num,transaction_time,amt,customer_prev_avg_amount,amount_deviation_ratio
0,2703186189652095,2019-01-01 00:00:18,4.97,NaN,NaN
1,630423337322,2019-01-01 00:00:44,107.23,NaN,NaN
2,38859492057661,2019-01-01 00:00:51,220.11,NaN,NaN
3,3534093764340240,2019-01-01 00:01:16,45.00,NaN,NaN
4,375534208663984,2019-01-01 00:03:06,41.96,NaN,NaN
5,4767265376804500,2019-01-01 00:04:08,94.63,NaN,NaN
6,30074693890476,2019-01-01 00:04:42,44.54,NaN,NaN
7,6011360759745864,2019-01-01 00:05:08,71.65,NaN,NaN
8,4922710831011201,2019-01-01 00:05:18,4.27,NaN,NaN
9,2720830304681674,2019-01-01 00:06:01,198.39,NaN,NaN


In [12]:
df["previous_transaction_time"] = (
    df.groupby("cc_num")["transaction_time"]
      .shift(1)
)

In [13]:
df["seconds_since_prev_transaction"] = (
    df["transaction_time"] -
    df["previous_transaction_time"]
).dt.total_seconds()

In [14]:
df = df.sort_values(["cc_num", "transaction_time"])

df["transactions_prev_1h"] = (
    df.set_index("transaction_time")
      .groupby("cc_num")["trans_num"]
      .rolling("1h")
      .count()
      .reset_index(level=0, drop=True)
      .reset_index(drop=True)
)

In [15]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arctan2(
        np.sqrt(a),
        np.sqrt(1 - a)
    )

In [16]:
df["customer_merchant_distance_km"] = haversine_distance(
    df["lat"],
    df["long"],
    df["merch_lat"],
    df["merch_long"]
)

In [17]:
df["merchant_seen_before"] = (
    df.groupby(["cc_num", "merchant"])
      .cumcount()
)

In [18]:
df["is_new_merchant"] = (
    df["merchant_seen_before"] == 0
).astype(int)

In [19]:
df["category_seen_before"] = (
    df.groupby(["cc_num", "category"])
      .cumcount()
)

In [20]:
df["is_new_category"] = (
    df["category_seen_before"] == 0
).astype(int)

In [21]:
df["customer_prev_count"] = df["customer_prev_count"].fillna(0)

In [22]:
df["has_customer_history"] = (
    df["customer_prev_count"] > 0
).astype(int)

## Leakage Checks

In [23]:
check_cols = [
    "cc_num",
    "transaction_time",
    "amt",
    "customer_prev_count",
    "customer_prev_avg_amount",
    "customer_prev_median_amount",
    "customer_prev_std_amount",
    "amount_deviation_ratio",
    "seconds_since_prev_transaction"
]

display(df[check_cols].head(20))

,cc_num,transaction_time,amt,customer_prev_count,customer_prev_avg_amount,customer_prev_median_amount,customer_prev_std_amount,amount_deviation_ratio,seconds_since_prev_transaction
1016,60416207185,2019-01-01 12:47:15,7.27,0,NaN,NaN,NaN,NaN,NaN
2724,60416207185,2019-01-02 08:44:57,52.94,1,7.270000,7.270,NaN,7.281981,71862.0
2726,60416207185,2019-01-02 08:47:36,82.08,2,30.105000,30.105,32.293567,2.726457,159.0
2882,60416207185,2019-01-02 12:38:14,34.79,3,47.430000,52.940,37.708144,0.733502,13838.0
2907,60416207185,2019-01-02 13:10:46,27.18,4,44.270000,43.865,31.430534,0.613960,1952.0
4135,60416207185,2019-01-03 13:56:35,6.87,5,40.852000,34.790,28.272292,0.168168,89149.0
4337,60416207185,2019-01-03 17:05:10,8.43,6,35.188333,30.985,28.843035,0.239568,11315.0
5467,60416207185,2019-01-04 13:59:55,117.11,7,31.365714,27.180,28.205570,3.733695,75285.0
6027,60416207185,2019-01-04 21:17:22,26.74,8,42.083750,30.985,40.011422,0.635400,26247.0
6273,60416207185,2019-01-05 00:42:24,105.20,9,40.378889,27.180,37.775106,2.605322,12302.0


In [24]:
customer_check = df[df["cc_num"] == df["cc_num"].iloc[10]].sort_values(
    "transaction_time"
)

display(
    customer_check[
        [
            "cc_num",
            "transaction_time",
            "amt",
            "customer_prev_count",
            "customer_prev_avg_amount",
            "amount_deviation_ratio",
            "seconds_since_prev_transaction"
        ]
    ].head(15)
)

,cc_num,transaction_time,amt,customer_prev_count,customer_prev_avg_amount,amount_deviation_ratio,seconds_since_prev_transaction
1016,60416207185,2019-01-01 12:47:15,7.27,0,NaN,NaN,NaN
2724,60416207185,2019-01-02 08:44:57,52.94,1,7.270000,7.281981,71862.0
2726,60416207185,2019-01-02 08:47:36,82.08,2,30.105000,2.726457,159.0
2882,60416207185,2019-01-02 12:38:14,34.79,3,47.430000,0.733502,13838.0
2907,60416207185,2019-01-02 13:10:46,27.18,4,44.270000,0.613960,1952.0
4135,60416207185,2019-01-03 13:56:35,6.87,5,40.852000,0.168168,89149.0
4337,60416207185,2019-01-03 17:05:10,8.43,6,35.188333,0.239568,11315.0
5467,60416207185,2019-01-04 13:59:55,117.11,7,31.365714,3.733695,75285.0
6027,60416207185,2019-01-04 21:17:22,26.74,8,42.083750,0.635400,26247.0
6273,60416207185,2019-01-05 00:42:24,105.20,9,40.378889,2.605322,12302.0


In [25]:
check_cols = [
    "cc_num",
    "transaction_time",
    "amt",
    "customer_prev_count",
    "customer_prev_avg_amount",
    "customer_prev_median_amount",
    "customer_prev_std_amount",
    "amount_deviation_ratio",
    "seconds_since_prev_transaction"
]

display(df[check_cols].head(15))

,cc_num,transaction_time,amt,customer_prev_count,customer_prev_avg_amount,customer_prev_median_amount,customer_prev_std_amount,amount_deviation_ratio,seconds_since_prev_transaction
1016,60416207185,2019-01-01 12:47:15,7.27,0,NaN,NaN,NaN,NaN,NaN
2724,60416207185,2019-01-02 08:44:57,52.94,1,7.270000,7.270,NaN,7.281981,71862.0
2726,60416207185,2019-01-02 08:47:36,82.08,2,30.105000,30.105,32.293567,2.726457,159.0
2882,60416207185,2019-01-02 12:38:14,34.79,3,47.430000,52.940,37.708144,0.733502,13838.0
2907,60416207185,2019-01-02 13:10:46,27.18,4,44.270000,43.865,31.430534,0.613960,1952.0
4135,60416207185,2019-01-03 13:56:35,6.87,5,40.852000,34.790,28.272292,0.168168,89149.0
4337,60416207185,2019-01-03 17:05:10,8.43,6,35.188333,30.985,28.843035,0.239568,11315.0
5467,60416207185,2019-01-04 13:59:55,117.11,7,31.365714,27.180,28.205570,3.733695,75285.0
6027,60416207185,2019-01-04 21:17:22,26.74,8,42.083750,30.985,40.011422,0.635400,26247.0
6273,60416207185,2019-01-05 00:42:24,105.20,9,40.378889,27.180,37.775106,2.605322,12302.0


In [26]:
customer_check = (
    df[df["cc_num"] == df["cc_num"].iloc[10]]
    .sort_values("transaction_time")
)

display(
    customer_check[
        [
            "transaction_time",
            "amt",
            "customer_prev_count",
            "customer_prev_avg_amount",
            "customer_prev_median_amount",
            "customer_prev_std_amount",
            "amount_deviation_ratio",
            "seconds_since_prev_transaction"
        ]
    ].head(15)
)

,transaction_time,amt,customer_prev_count,customer_prev_avg_amount,customer_prev_median_amount,customer_prev_std_amount,amount_deviation_ratio,seconds_since_prev_transaction
1016,2019-01-01 12:47:15,7.27,0,NaN,NaN,NaN,NaN,NaN
2724,2019-01-02 08:44:57,52.94,1,7.270000,7.270,NaN,7.281981,71862.0
2726,2019-01-02 08:47:36,82.08,2,30.105000,30.105,32.293567,2.726457,159.0
2882,2019-01-02 12:38:14,34.79,3,47.430000,52.940,37.708144,0.733502,13838.0
2907,2019-01-02 13:10:46,27.18,4,44.270000,43.865,31.430534,0.613960,1952.0
4135,2019-01-03 13:56:35,6.87,5,40.852000,34.790,28.272292,0.168168,89149.0
4337,2019-01-03 17:05:10,8.43,6,35.188333,30.985,28.843035,0.239568,11315.0
5467,2019-01-04 13:59:55,117.11,7,31.365714,27.180,28.205570,3.733695,75285.0
6027,2019-01-04 21:17:22,26.74,8,42.083750,30.985,40.011422,0.635400,26247.0
6273,2019-01-05 00:42:24,105.20,9,40.378889,27.180,37.775106,2.605322,12302.0


In [27]:
print(
    df.groupby("cc_num")["transaction_time"]
      .apply(lambda x: x.is_monotonic_increasing)
      .value_counts()
)

transaction_time
True    983
Name: count, dtype: int64


In [28]:
display(
    customer_check[
        [
            "transaction_time",
            "amt",
            "customer_prev_count",
            "customer_prev_avg_amount",
            "customer_prev_median_amount",
            "customer_prev_std_amount",
            "amount_deviation_ratio",
            "seconds_since_prev_transaction"
        ]
    ].head(15)
)

,transaction_time,amt,customer_prev_count,customer_prev_avg_amount,customer_prev_median_amount,customer_prev_std_amount,amount_deviation_ratio,seconds_since_prev_transaction
1016,2019-01-01 12:47:15,7.27,0,NaN,NaN,NaN,NaN,NaN
2724,2019-01-02 08:44:57,52.94,1,7.270000,7.270,NaN,7.281981,71862.0
2726,2019-01-02 08:47:36,82.08,2,30.105000,30.105,32.293567,2.726457,159.0
2882,2019-01-02 12:38:14,34.79,3,47.430000,52.940,37.708144,0.733502,13838.0
2907,2019-01-02 13:10:46,27.18,4,44.270000,43.865,31.430534,0.613960,1952.0
4135,2019-01-03 13:56:35,6.87,5,40.852000,34.790,28.272292,0.168168,89149.0
4337,2019-01-03 17:05:10,8.43,6,35.188333,30.985,28.843035,0.239568,11315.0
5467,2019-01-04 13:59:55,117.11,7,31.365714,27.180,28.205570,3.733695,75285.0
6027,2019-01-04 21:17:22,26.74,8,42.083750,30.985,40.011422,0.635400,26247.0
6273,2019-01-05 00:42:24,105.20,9,40.378889,27.180,37.775106,2.605322,12302.0


In [29]:
df["_row_id"] = np.arange(len(df))

In [30]:
df = df.sort_values(
    ["cc_num", "transaction_time"]
).reset_index(drop=True)

In [31]:
velocity_1h = (
    df.set_index("transaction_time")
      .groupby("cc_num")["trans_num"]
      .rolling("1h", closed="left")
      .count()
      .reset_index()
)

velocity_1h.columns = [
    "cc_num",
    "transaction_time",
    "transactions_prev_1h"
]


In [32]:
df = df.merge(
    velocity_1h,
    on=["cc_num", "transaction_time"],
    how="left"
)

In [33]:
same_time = (
    df.groupby(["cc_num", "transaction_time"])
      .size()
      .reset_index(name="count")
)

print(
    "Maximum transactions for same customer at same timestamp:",
    same_time["count"].max()
)

Maximum transactions for same customer at same timestamp: 4


In [34]:
# Make sure transactions are ordered
df = df.sort_values(
    ["cc_num", "transaction_time"]
).reset_index(drop=True)

# Create a temporary unique row identifier
df["_row_id"] = np.arange(len(df))

# Calculate previous 1-hour transaction count
velocity_1h = (
    df.groupby("cc_num", group_keys=False)
      .apply(
          lambda g: g.assign(
              transactions_prev_1h=
              g.rolling(
                  "1h",
                  on="transaction_time",
                  closed="left"
              )["trans_num"].count()
          ),
          include_groups=False
      )
      .reset_index(drop=True)
)

In [35]:
print("transactions_prev_1h" in df.columns)

print(
    "Maximum transactions for same customer at same timestamp:",
    df.groupby(["cc_num", "transaction_time"]).size().max()
)

False
Maximum transactions for same customer at same timestamp: 4


In [36]:
df["_row_id"] = np.arange(len(df))

In [37]:
# Make sure transactions are chronologically ordered
df = (
    df.sort_values(["cc_num", "transaction_time", "_row_id"])
      .reset_index(drop=True)
)

# Calculate previous 1-hour transaction count for each transaction
velocity_1h = (
    df.groupby("cc_num", group_keys=False)
      .apply(
          lambda g: pd.DataFrame({
              "_row_id": g["_row_id"].to_numpy(),

              "transactions_prev_1h": (
                  g.set_index("transaction_time")["trans_num"]
                   .rolling("1h", closed="left")
                   .count()
                   .to_numpy()
              )
          }),
          include_groups=False
      )
      .reset_index(drop=True)
)

print(velocity_1h.head())

   _row_id  transactions_prev_1h
0        0                   NaN
1        1                   NaN
2        2                   1.0
3        3                   NaN
4        4                   1.0


In [38]:
df = df.merge(
    velocity_1h,
    on="_row_id",
    how="left"
)

df["transactions_prev_1h"] = (
    df["transactions_prev_1h"].fillna(0)
)

In [39]:
print(
    "Column exists:",
    "transactions_prev_1h" in df.columns
)

Column exists: True


In [40]:
display(
    df[
        [
            "_row_id",
            "cc_num",
            "transaction_time",
            "amt",
            "transactions_prev_1h"
        ]
    ].head(20)
)

,_row_id,cc_num,transaction_time,amt,transactions_prev_1h
0,0,60416207185,2019-01-01 12:47:15,7.27,0.0
1,1,60416207185,2019-01-02 08:44:57,52.94,0.0
2,2,60416207185,2019-01-02 08:47:36,82.08,1.0
3,3,60416207185,2019-01-02 12:38:14,34.79,0.0
4,4,60416207185,2019-01-02 13:10:46,27.18,1.0
5,5,60416207185,2019-01-03 13:56:35,6.87,0.0
6,6,60416207185,2019-01-03 17:05:10,8.43,0.0
7,7,60416207185,2019-01-04 13:59:55,117.11,0.0
8,8,60416207185,2019-01-04 21:17:22,26.74,0.0
9,9,60416207185,2019-01-05 00:42:24,105.20,0.0


In [41]:
print(df["transactions_prev_1h"].describe())

count    1.296715e+06
mean     1.895420e-01
std      4.618415e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      6.000000e+00
Name: transactions_prev_1h, dtype: float64


In [42]:
print(
    "Maximum previous 1-hour transactions:",
    df["transactions_prev_1h"].max()
)

Maximum previous 1-hour transactions: 6.0


In [43]:
customer_velocity_check = (
    df[df["cc_num"] == df["cc_num"].iloc[10]]
    .sort_values(["transaction_time", "_row_id"])
)

display(
    customer_velocity_check[
        [
            "transaction_time",
            "amt",
            "transactions_prev_1h"
        ]
    ].head(20)
)

,transaction_time,amt,transactions_prev_1h
0,2019-01-01 12:47:15,7.27,0.0
1,2019-01-02 08:44:57,52.94,0.0
2,2019-01-02 08:47:36,82.08,1.0
3,2019-01-02 12:38:14,34.79,0.0
4,2019-01-02 13:10:46,27.18,1.0
5,2019-01-03 13:56:35,6.87,0.0
6,2019-01-03 17:05:10,8.43,0.0
7,2019-01-04 13:59:55,117.11,0.0
8,2019-01-04 21:17:22,26.74,0.0
9,2019-01-05 00:42:24,105.20,0.0


In [44]:
print("Maximum previous 1-hour transactions:",
      df["transactions_prev_1h"].max())

Maximum previous 1-hour transactions: 6.0


In [45]:
print(df["transactions_prev_1h"].describe())

count    1.296715e+06
mean     1.895420e-01
std      4.618415e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      6.000000e+00
Name: transactions_prev_1h, dtype: float64


In [46]:
velocity_5min = (
    df.groupby("cc_num", group_keys=False)
      .apply(
          lambda g: pd.DataFrame({
              "_row_id": g["_row_id"].to_numpy(),
              "transactions_prev_5min": (
                  g.set_index("transaction_time")["trans_num"]
                   .rolling("5min", closed="left")
                   .count()
                   .to_numpy()
              )
          }),
          include_groups=False
      )
      .reset_index(drop=True)
)

df = df.merge(
    velocity_5min,
    on="_row_id",
    how="left"
)

df["transactions_prev_5min"] = (
    df["transactions_prev_5min"].fillna(0)
)

In [47]:
velocity_24h = (
    df.groupby("cc_num", group_keys=False)
      .apply(
          lambda g: pd.DataFrame({
              "_row_id": g["_row_id"].to_numpy(),
              "transactions_prev_24h": (
                  g.set_index("transaction_time")["trans_num"]
                   .rolling("24h", closed="left")
                   .count()
                   .to_numpy()
              )
          }),
          include_groups=False
      )
      .reset_index(drop=True)
)

df = df.merge(
    velocity_24h,
    on="_row_id",
    how="left"
)

df["transactions_prev_24h"] = (
    df["transactions_prev_24h"].fillna(0)
)

In [48]:
velocity_cols = [
    "transactions_prev_5min",
    "transactions_prev_1h",
    "transactions_prev_24h"
]

display(
    df[
        [
            "transaction_time",
            "amt",
            *velocity_cols
        ]
    ].head(20)
)

,transaction_time,amt,transactions_prev_5min,transactions_prev_1h,transactions_prev_24h
0,2019-01-01 12:47:15,7.27,0.0,0.0,0.0
1,2019-01-02 08:44:57,52.94,0.0,0.0,1.0
2,2019-01-02 08:47:36,82.08,1.0,1.0,2.0
3,2019-01-02 12:38:14,34.79,0.0,0.0,3.0
4,2019-01-02 13:10:46,27.18,0.0,1.0,3.0
5,2019-01-03 13:56:35,6.87,0.0,0.0,0.0
6,2019-01-03 17:05:10,8.43,0.0,0.0,1.0
7,2019-01-04 13:59:55,117.11,0.0,0.0,1.0
8,2019-01-04 21:17:22,26.74,0.0,0.0,1.0
9,2019-01-05 00:42:24,105.20,0.0,0.0,2.0


In [49]:
df[velocity_cols].describe()

,transactions_prev_5min,transactions_prev_1h,transactions_prev_24h
count,1.296715e+06,1.296715e+06,1.296715e+06
mean,1.600120e-02,1.895420e-01,3.884359e+00
std,1.272375e-01,4.618415e-01,3.082387e+00
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,0.000000e+00,0.000000e+00,2.000000e+00
50%,0.000000e+00,0.000000e+00,3.000000e+00
75%,0.000000e+00,0.000000e+00,5.000000e+00
max,4.000000e+00,6.000000e+00,3.500000e+01


In [50]:
velocity_cols = [
    "transactions_prev_5min",
    "transactions_prev_1h",
    "transactions_prev_24h"
]

display(
    df[
        [
            "transaction_time",
            "cc_num",
            "amt",
            "is_fraud",
            *velocity_cols
        ]
    ]
    .sort_values(
        ["transactions_prev_1h", "transactions_prev_5min"],
        ascending=False
    )
    .head(20)
)

,transaction_time,cc_num,amt,is_fraud,transactions_prev_5min,transactions_prev_1h,transactions_prev_24h
828591,2020-06-05 13:03:12,3596217206093829,9.14,0,2.0,6.0,12.0
49776,2019-12-16 15:56:37,630423337322,19.01,0,1.0,6.0,20.0
243455,2019-12-22 16:17:13,30273037698427,4.79,0,1.0,6.0,21.0
472741,2019-12-01 16:41:53,372520049757633,59.78,0,1.0,6.0,21.0
697587,2020-06-08 15:45:05,3541160328600277,69.37,0,1.0,6.0,16.0
783517,2019-12-29 21:30:14,3576431665303017,6.14,0,1.0,6.0,16.0
1194091,2020-03-15 23:08:46,4208110975550360171,857.88,1,1.0,6.0,8.0
42275,2019-08-26 22:29:42,581686439828,5.20,0,0.0,6.0,14.0
79430,2019-07-22 23:39:37,676148621961,674.94,1,0.0,6.0,11.0
243456,2019-12-22 16:27:16,30273037698427,49.74,0,0.0,6.0,21.0


In [51]:
velocity_fraud = (
    df.groupby("transactions_prev_1h")["is_fraud"]
      .agg(
          transaction_count="count",
          fraud_count="sum",
          fraud_rate="mean"
      )
)

velocity_fraud

,transaction_count,fraud_count,fraud_rate
transactions_prev_1h,,,
0.0,1084166,4240,0.003911
1.0,183916,2023,0.011000
2.0,24757,822,0.033203
3.0,3273,292,0.089215
4.0,499,101,0.202405
5.0,87,25,0.287356
6.0,17,3,0.176471


In [52]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arctan2(
        np.sqrt(a),
        np.sqrt(1 - a)
    )

In [53]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arctan2(
        np.sqrt(a),
        np.sqrt(1 - a)
    )

In [54]:
df["customer_merchant_distance_km"] = haversine_distance(
    df["lat"],
    df["long"],
    df["merch_lat"],
    df["merch_long"]
)

In [55]:
print(df["customer_merchant_distance_km"].describe())

count    1.296715e+06
mean     7.611470e+01
std      2.911693e+01
min      2.225452e-02
25%      5.533493e+01
50%      7.823202e+01
75%      9.850332e+01
max      1.521172e+02
Name: customer_merchant_distance_km, dtype: float64


In [56]:
distance_stats = (
    df.groupby("is_fraud")["customer_merchant_distance_km"]
      .describe()
)

distance_stats

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,1289209.0,76.113808,29.119047,0.022255,55.332853,78.233026,98.504563,152.117173
1,7506.0,76.268330,28.752602,0.738769,55.632890,77.931954,98.391090,144.522410


In [57]:
distance_median = (
    df.groupby("is_fraud")["customer_merchant_distance_km"]
      .median()
)

distance_median

is_fraud
0    78.233026
1    77.931954
Name: customer_merchant_distance_km, dtype: float64

In [58]:
df["merchant_seen_before"] = (
    df.groupby(["cc_num", "merchant"])
      .cumcount()
)

df["is_new_merchant"] = (
    df["merchant_seen_before"] == 0
).astype(int)

In [59]:
merchant_new_fraud = (
    df.groupby("is_new_merchant")["is_fraud"]
      .agg(
          transaction_count="count",
          fraud_count="sum",
          fraud_rate="mean"
      )
)

merchant_new_fraud

,transaction_count,fraud_count,fraud_rate
is_new_merchant,,,
0,817643,3586,0.004386
1,479072,3920,0.008182


In [60]:
df["category_seen_before"] = (
    df.groupby(["cc_num", "category"])
      .cumcount()
)

df["is_new_category"] = (
    df["category_seen_before"] == 0
).astype(int)

In [61]:
category_new_fraud = (
    df.groupby("is_new_category")["is_fraud"]
      .agg(
          transaction_count="count",
          fraud_count="sum",
          fraud_rate="mean"
      )
)

category_new_fraud

,transaction_count,fraud_count,fraud_rate
is_new_category,,,
0,1283627,7042,0.005486
1,13088,464,0.035452


In [62]:
feature_summary = (
    df.groupby("is_fraud")
      .agg(
          avg_amount=("amt", "mean"),
          avg_amount_deviation=("amount_deviation_ratio", "mean"),
          avg_prev_5min=("transactions_prev_5min", "mean"),
          avg_prev_1h=("transactions_prev_1h", "mean"),
          avg_prev_24h=("transactions_prev_24h", "mean"),
          avg_distance_km=("customer_merchant_distance_km", "mean"),
          new_merchant_rate=("is_new_merchant", "mean"),
          new_category_rate=("is_new_category", "mean")
      )
)

feature_summary

,avg_amount,avg_amount_deviation,avg_prev_5min,avg_prev_1h,avg_prev_24h,avg_distance_km,new_merchant_rate,new_category_rate
is_fraud,,,,,,,,
0,67.666348,0.986716,0.015698,0.186697,3.882327,76.113808,0.368561,0.009792
1,531.320092,6.983198,0.068079,0.678124,4.233413,76.268330,0.522249,0.061817


In [63]:
engineered_features = [
    "customer_prev_count",
    "customer_prev_avg_amount",
    "customer_prev_median_amount",
    "customer_prev_std_amount",
    "amount_deviation_ratio",
    "seconds_since_prev_transaction",
    "transactions_prev_5min",
    "transactions_prev_1h",
    "transactions_prev_24h",
    "customer_merchant_distance_km",
    "is_new_merchant",
    "is_new_category"
]

feature_quality = pd.DataFrame({
    "missing_count": df[engineered_features].isna().sum(),
    "missing_pct": df[engineered_features].isna().mean() * 100,
    "dtype": df[engineered_features].dtypes.astype(str)
})

feature_quality

,missing_count,missing_pct,dtype
customer_prev_count,0,0.000000,int64
customer_prev_avg_amount,983,0.075807,float64
customer_prev_median_amount,983,0.075807,float64
customer_prev_std_amount,1966,0.151614,float64
amount_deviation_ratio,983,0.075807,float64
seconds_since_prev_transaction,983,0.075807,float64
transactions_prev_5min,0,0.000000,float64
transactions_prev_1h,0,0.000000,float64
transactions_prev_24h,0,0.000000,float64
customer_merchant_distance_km,0,0.000000,float64


In [64]:
print(
    "Negative amount:",
    (df["amt"] < 0).sum()
)

print(
    "Negative distance:",
    (df["customer_merchant_distance_km"] < 0).sum()
)

print(
    "Negative velocity:",
    (df["transactions_prev_1h"] < 0).sum()
)

Negative amount: 0
Negative distance: 0
Negative velocity: 0


In [65]:
MODEL_FEATURES = [
    "amt",
    "log_amount",
    "hour",
    "day_of_week",
    "month",
    "day_of_month",
    "is_weekend",
    "customer_prev_count",
    "customer_prev_avg_amount",
    "customer_prev_median_amount",
    "customer_prev_std_amount",
    "amount_deviation_ratio",
    "seconds_since_prev_transaction",
    "transactions_prev_5min",
    "transactions_prev_1h",
    "transactions_prev_24h",
    "merchant_seen_before",
    "is_new_merchant",
    "category_seen_before",
    "is_new_category",
    "customer_merchant_distance_km"
]

TARGET = "is_fraud"

In [66]:
processed_df = df[
    ["transaction_time", "cc_num", "trans_num"] +
    MODEL_FEATURES +
    [TARGET]
].copy()

print("Processed shape:", processed_df.shape)

display(processed_df.head())

Processed shape: (1296715, 25)


,transaction_time,cc_num,trans_num,amt,log_amount,hour,day_of_week,month,day_of_month,is_weekend,...,seconds_since_prev_transaction,transactions_prev_5min,transactions_prev_1h,transactions_prev_24h,merchant_seen_before,is_new_merchant,category_seen_before,is_new_category,customer_merchant_distance_km,is_fraud
0,2019-01-01 12:47:15,60416207185,98e3dcf98101146a577f85a34e58feec,7.27,2.112635,12,1,1,1,0,...,NaN,0.0,0.0,0.0,0,1,0,1,127.606239,0
1,2019-01-02 08:44:57,60416207185,498120fc45d277f7c88e3dba79c33865,52.94,3.987872,8,2,1,2,0,...,71862.0,0.0,0.0,1.0,0,1,0,1,110.308921,0
2,2019-01-02 08:47:36,60416207185,95f514bb993151347c7acdf8505c3d62,82.08,4.419804,8,2,1,2,0,...,159.0,1.0,1.0,2.0,0,1,1,0,21.787261,0
3,2019-01-02 12:38:14,60416207185,4f0c1a14e0aa7eb56a490780ef9268c5,34.79,3.577669,12,2,1,2,0,...,13838.0,0.0,0.0,3.0,0,1,0,1,87.204215,0
4,2019-01-02 13:10:46,60416207185,3b2ebd3af508afba959640893e1e82bc,27.18,3.338613,13,2,1,2,0,...,1952.0,0.0,1.0,3.0,0,1,0,1,74.212965,0


In [67]:
processed_df.to_parquet(
    "../data/processed/sparkov_features.parquet",
    index=False
)

In [68]:
from pathlib import Path

output_path = Path(
    "../data/processed/sparkov_features.parquet"
)

print("Exists:", output_path.exists())
print(
    "Size (MB):",
    round(output_path.stat().st_size / (1024 * 1024), 2)
)

Exists: True
Size (MB): 109.07


In [69]:
test_df = pd.read_parquet(
    "../data/processed/sparkov_features.parquet"
)

print("Shape:", test_df.shape)
display(test_df.head())

Shape: (1296715, 25)


,transaction_time,cc_num,trans_num,amt,log_amount,hour,day_of_week,month,day_of_month,is_weekend,...,seconds_since_prev_transaction,transactions_prev_5min,transactions_prev_1h,transactions_prev_24h,merchant_seen_before,is_new_merchant,category_seen_before,is_new_category,customer_merchant_distance_km,is_fraud
0,2019-01-01 12:47:15,60416207185,98e3dcf98101146a577f85a34e58feec,7.27,2.112635,12,1,1,1,0,...,NaN,0.0,0.0,0.0,0,1,0,1,127.606239,0
1,2019-01-02 08:44:57,60416207185,498120fc45d277f7c88e3dba79c33865,52.94,3.987872,8,2,1,2,0,...,71862.0,0.0,0.0,1.0,0,1,0,1,110.308921,0
2,2019-01-02 08:47:36,60416207185,95f514bb993151347c7acdf8505c3d62,82.08,4.419804,8,2,1,2,0,...,159.0,1.0,1.0,2.0,0,1,1,0,21.787261,0
3,2019-01-02 12:38:14,60416207185,4f0c1a14e0aa7eb56a490780ef9268c5,34.79,3.577669,12,2,1,2,0,...,13838.0,0.0,0.0,3.0,0,1,0,1,87.204215,0
4,2019-01-02 13:10:46,60416207185,3b2ebd3af508afba959640893e1e82bc,27.18,3.338613,13,2,1,2,0,...,1952.0,0.0,1.0,3.0,0,1,0,1,74.212965,0


In [70]:
print("Rows in engineered df:", len(df))
print("Unique transaction IDs:", df["trans_num"].nunique())
print("Duplicate transaction IDs:", df["trans_num"].duplicated().sum())
print("Unique row IDs:", df["_row_id"].nunique())
print("Duplicate row IDs:", df["_row_id"].duplicated().sum())

Rows in engineered df: 1296715
Unique transaction IDs: 1296675
Duplicate transaction IDs: 40
Unique row IDs: 1296715
Duplicate row IDs: 0


In [71]:
duplicates = (
    df[df["trans_num"].duplicated(keep=False)]
      .sort_values(["trans_num", "_row_id"])
)

print("Duplicated transaction IDs:",
      duplicates["trans_num"].nunique())

display(
    duplicates[
        [
            "_row_id",
            "trans_num",
            "cc_num",
            "transaction_time",
            "amt",
            "is_fraud"
        ]
    ].head(100)
)

Duplicated transaction IDs: 40


,_row_id,trans_num,cc_num,transaction_time,amt,is_fraud
487959,487959,008f89cdba4643cfa89529e602d2a9a8,375082648741747,2019-06-28 21:25:50,2.66,0
487960,487960,008f89cdba4643cfa89529e602d2a9a8,375082648741747,2019-06-28 21:25:50,2.66,0
1261821,1261821,059c4159d4a26fe01bbf6f07749a903b,4736845434667908128,2019-12-25 00:01:08,117.50,0
1261822,1261822,059c4159d4a26fe01bbf6f07749a903b,4736845434667908128,2019-12-25 00:01:08,117.50,0
720083,720083,165f8c7e631d3513459979bfd13a95df,3553629419254918,2019-03-10 02:31:27,90.73,0
...,...,...,...,...,...,...
487962,487962,f211adb01a1a37d910e95f4cd38768c7,375082648741747,2019-06-28 21:25:50,5.68,0
736258,736258,f9efd36d6f5c3ec2b004feb97baa7c25,3560318482131952,2019-10-31 01:16:14,26.88,0
736259,736259,f9efd36d6f5c3ec2b004feb97baa7c25,3560318482131952,2019-10-31 01:16:14,26.88,0
81098,81098,fb277314380561f3fdca64ac19d80c8a,676173792455,2019-12-14 01:11:03,49.25,0


In [72]:
print(
    duplicates.groupby("trans_num")["_row_id"]
    .count()
    .value_counts()
)

_row_id
2    40
Name: count, dtype: int64


In [73]:
print(
    "velocity_1h:",
    len(velocity_1h),
    "rows | duplicate _row_id:",
    velocity_1h["_row_id"].duplicated().sum()
)

print(
    "velocity_5min:",
    len(velocity_5min),
    "rows | duplicate _row_id:",
    velocity_5min["_row_id"].duplicated().sum()
)

print(
    "velocity_24h:",
    len(velocity_24h),
    "rows | duplicate _row_id:",
    velocity_24h["_row_id"].duplicated().sum()
)


velocity_1h: 1296715 rows | duplicate _row_id: 0
velocity_5min: 1296715 rows | duplicate _row_id: 0
velocity_24h: 1296715 rows | duplicate _row_id: 0


## Clean Feature Pipeline Rebuild

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

RAW_PATH = Path("../data/raw/sparkov/fraudTrain.csv")

clean_df = pd.read_csv(RAW_PATH)

clean_df = clean_df.drop(
    columns=["Unnamed: 0"],
    errors="ignore"
)

clean_df["transaction_time"] = pd.to_datetime(
    clean_df["trans_date_trans_time"]
)

clean_df = (
    clean_df
    .sort_values(["cc_num", "transaction_time"])
    .reset_index(drop=True)
)

clean_df["_row_id"] = np.arange(len(clean_df))

print("Initial rows:", len(clean_df))
print("Unique transactions:", clean_df["trans_num"].nunique())
print(
    "Duplicate transactions:",
    clean_df["trans_num"].duplicated().sum()
)
print("Unique row IDs:", clean_df["_row_id"].nunique())

Initial rows: 1296675
Unique transactions: 1296675
Duplicate transactions: 0
Unique row IDs: 1296675


In [2]:
clean_df["hour"] = clean_df["transaction_time"].dt.hour
clean_df["day_of_week"] = clean_df["transaction_time"].dt.dayofweek
clean_df["month"] = clean_df["transaction_time"].dt.month
clean_df["day_of_month"] = clean_df["transaction_time"].dt.day
clean_df["is_weekend"] = (
    clean_df["day_of_week"].isin([5, 6]).astype(int)
)

clean_df["log_amount"] = np.log1p(clean_df["amt"])

In [3]:
clean_df["customer_prev_count"] = (
    clean_df.groupby("cc_num").cumcount()
)

clean_df["customer_prev_avg_amount"] = (
    clean_df.groupby("cc_num")["amt"]
    .transform(lambda x: x.shift().expanding().mean())
)

clean_df["customer_prev_median_amount"] = (
    clean_df.groupby("cc_num")["amt"]
    .transform(lambda x: x.shift().expanding().median())
)

clean_df["customer_prev_std_amount"] = (
    clean_df.groupby("cc_num")["amt"]
    .transform(lambda x: x.shift().expanding().std())
)

clean_df["amount_deviation_ratio"] = (
    clean_df["amt"] /
    clean_df["customer_prev_avg_amount"]
)

clean_df["previous_transaction_time"] = (
    clean_df.groupby("cc_num")["transaction_time"].shift()
)

clean_df["seconds_since_prev_transaction"] = (
    clean_df["transaction_time"] -
    clean_df["previous_transaction_time"]
).dt.total_seconds()

In [4]:
print("Rows after non-merge features:", len(clean_df))
print(
    "Duplicate transactions:",
    clean_df["trans_num"].duplicated().sum()
)

Rows after non-merge features: 1296675
Duplicate transactions: 0


## Clean Velocity Feature Rebuild

In [5]:
velocity_1h_clean = (
    clean_df.groupby("cc_num", group_keys=False)
    .apply(
        lambda g: pd.DataFrame({
            "_row_id": g["_row_id"].to_numpy(),
            "transactions_prev_1h": (
                g.set_index("transaction_time")["trans_num"]
                .rolling("1h", closed="left")
                .count()
                .to_numpy()
            )
        }),
        include_groups=False
    )
    .reset_index(drop=True)
)

In [6]:
print("Velocity 1h rows:", len(velocity_1h_clean))
print(
    "Velocity 1h duplicate row IDs:",
    velocity_1h_clean["_row_id"].duplicated().sum()
)

Velocity 1h rows: 1296675
Velocity 1h duplicate row IDs: 0


In [7]:
clean_df = clean_df.merge(
    velocity_1h_clean,
    on="_row_id",
    how="left",
    validate="one_to_one"
)

In [8]:
print("Rows after 1h merge:", len(clean_df))
print(
    "Duplicate transactions after 1h:",
    clean_df["trans_num"].duplicated().sum()
)

Rows after 1h merge: 1296675
Duplicate transactions after 1h: 0


In [9]:
velocity_5min_clean = (
    clean_df.groupby("cc_num", group_keys=False)
    .apply(
        lambda g: pd.DataFrame({
            "_row_id": g["_row_id"].to_numpy(),
            "transactions_prev_5min": (
                g.set_index("transaction_time")["trans_num"]
                .rolling("5min", closed="left")
                .count()
                .to_numpy()
            )
        }),
        include_groups=False
    )
    .reset_index(drop=True)
)

In [10]:
print(
    "Velocity 5min rows:",
    len(velocity_5min_clean)
)

print(
    "Velocity 5min duplicate row IDs:",
    velocity_5min_clean["_row_id"].duplicated().sum()
)

Velocity 5min rows: 1296675
Velocity 5min duplicate row IDs: 0


In [11]:
clean_df = clean_df.merge(
    velocity_5min_clean,
    on="_row_id",
    how="left",
    validate="one_to_one"
)

In [12]:
print(
    "Rows after 5min merge:",
    len(clean_df)
)

print(
    "Duplicate transactions after 5min:",
    clean_df["trans_num"].duplicated().sum()
)

Rows after 5min merge: 1296675
Duplicate transactions after 5min: 0


In [13]:
velocity_24h_clean = (
    clean_df.groupby("cc_num", group_keys=False)
    .apply(
        lambda g: pd.DataFrame({
            "_row_id": g["_row_id"].to_numpy(),
            "transactions_prev_24h": (
                g.set_index("transaction_time")["trans_num"]
                .rolling("24h", closed="left")
                .count()
                .to_numpy()
            )
        }),
        include_groups=False
    )
    .reset_index(drop=True)
)

In [14]:
print(
    "Velocity 24h rows:",
    len(velocity_24h_clean)
)

print(
    "Velocity 24h duplicate row IDs:",
    velocity_24h_clean["_row_id"].duplicated().sum()
)

Velocity 24h rows: 1296675
Velocity 24h duplicate row IDs: 0


In [15]:
clean_df = clean_df.merge(
    velocity_24h_clean,
    on="_row_id",
    how="left",
    validate="one_to_one"
)

In [16]:
print(
    "Rows after 24h merge:",
    len(clean_df)
)

print(
    "Duplicate transactions after 24h:",
    clean_df["trans_num"].duplicated().sum()
)

Rows after 24h merge: 1296675
Duplicate transactions after 24h: 0


In [17]:
print("Final rows:", len(clean_df))
print("Unique transactions:", clean_df["trans_num"].nunique())
print("Duplicate transactions:",
      clean_df["trans_num"].duplicated().sum())
print("Unique row IDs:", clean_df["_row_id"].nunique())
print("Duplicate row IDs:", clean_df["_row_id"].duplicated().sum())

Final rows: 1296675
Unique transactions: 1296675
Duplicate transactions: 0
Unique row IDs: 1296675
Duplicate row IDs: 0


In [18]:
MODEL_FEATURES = [
    "amt",
    "log_amount",
    "hour",
    "day_of_week",
    "month",
    "day_of_month",
    "is_weekend",

    "customer_prev_count",
    "customer_prev_avg_amount",
    "customer_prev_median_amount",
    "customer_prev_std_amount",
    "amount_deviation_ratio",
    "seconds_since_prev_transaction",

    "transactions_prev_5min",
    "transactions_prev_1h",
    "transactions_prev_24h",

    "merchant_seen_before",
    "is_new_merchant",
    "category_seen_before",
    "is_new_category",

    "customer_merchant_distance_km"
]

TARGET = "is_fraud"

In [20]:
clean_df["merchant_seen_before"] = (
    clean_df.groupby(["cc_num", "merchant"]).cumcount()
)

clean_df["is_new_merchant"] = (
    clean_df["merchant_seen_before"] == 0
).astype(int)

In [21]:
clean_df["category_seen_before"] = (
    clean_df.groupby(["cc_num", "category"]).cumcount()
)

clean_df["is_new_category"] = (
    clean_df["category_seen_before"] == 0
).astype(int)

In [22]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arctan2(
        np.sqrt(a),
        np.sqrt(1 - a)
    )

clean_df["customer_merchant_distance_km"] = (
    haversine_distance(
        clean_df["lat"],
        clean_df["long"],
        clean_df["merch_lat"],
        clean_df["merch_long"]
    )
)

In [23]:
required_features = [
    "amt",
    "log_amount",
    "hour",
    "day_of_week",
    "month",
    "day_of_month",
    "is_weekend",
    "customer_prev_count",
    "customer_prev_avg_amount",
    "customer_prev_median_amount",
    "customer_prev_std_amount",
    "amount_deviation_ratio",
    "seconds_since_prev_transaction",
    "transactions_prev_5min",
    "transactions_prev_1h",
    "transactions_prev_24h",
    "merchant_seen_before",
    "is_new_merchant",
    "category_seen_before",
    "is_new_category",
    "customer_merchant_distance_km"
]

missing = [
    col for col in required_features
    if col not in clean_df.columns
]

print("Missing features:", missing)

Missing features: []


In [24]:
MODEL_FEATURES = required_features

TARGET = "is_fraud"

processed_df = clean_df[
    ["transaction_time", "cc_num", "trans_num"] +
    MODEL_FEATURES +
    [TARGET]
].copy()

print("Processed shape:", processed_df.shape)
print("Unique transactions:", processed_df["trans_num"].nunique())
print(
    "Duplicate transactions:",
    processed_df["trans_num"].duplicated().sum()
)

Processed shape: (1296675, 25)
Unique transactions: 1296675
Duplicate transactions: 0


In [25]:
processed_df.to_parquet(
    "../data/processed/sparkov_features.parquet",
    index=False
)

print("Processed dataset saved successfully.")

Processed dataset saved successfully.


In [26]:
test_df = pd.read_parquet(
    "../data/processed/sparkov_features.parquet"
)

print("Shape:", test_df.shape)
print("Unique transactions:", test_df["trans_num"].nunique())
print(
    "Duplicate transactions:",
    test_df["trans_num"].duplicated().sum()
)

Shape: (1296675, 25)
Unique transactions: 1296675
Duplicate transactions: 0
